In [1]:
import mysql.connector
from getpass import getpass

def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password=PASSWORD,
        database="bank_dbb"
    )

def create_account(name, opening_balance=0):
    if not name.strip():
        print("Name khali nahi ho sakta.")
        return
    if opening_balance < 0:
        print("Opening balance negative nahi ho sakta.")
        return
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO accounts (name, balance) VALUES (%s, %s)",
            (name.strip(), opening_balance)
        )
        conn.commit()
        print("Account ban gaya. Account number:", cursor.lastrowid)
    except mysql.connector.Error as e:
        conn.rollback()
        print("Error:", e)
    finally:
        conn.close()

def view_account(account_no):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT account_no, name, balance FROM accounts WHERE account_no = %s",
            (account_no,)
        )
        row = cursor.fetchone()
        if row is None:
            print("Ye account exist nahi karta.")
        else:
            print("Account No:", row[0], "| Name:", row[1], "| Balance:", row[2])
    finally:
        conn.close()

PASSWORD = getpass("MySQL password: ")
view_account(1)

MySQL password:  ········


Account No: 1 | Name: Test User | Balance: 1400.00


In [5]:
from decimal import Decimal
def deposit(account_no, amount):
    amount = Decimal(str(amount))
    if amount <= 0:
        print("Amount 0 se zyada hona chahiye.")
        return
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT balance FROM accounts WHERE account_no = %s FOR UPDATE",
            (account_no,)
        )
        if cursor.fetchone() is None:
            print("Ye account exist nahi karta.")
            return
        cursor.execute(
            "UPDATE accounts SET balance = balance + %s WHERE account_no = %s",
            (amount, account_no)
        )
        cursor.execute(
            "INSERT INTO transactions (account_no, type, amount) VALUES (%s, 'deposit', %s)",
            (account_no, amount)
        )
        conn.commit()
        print("Deposit ho gaya.")
    except mysql.connector.Error as e:
        conn.rollback()
        print("Error:", e)
    finally:
        conn.close()

def withdraw(account_no, amount):
    amount = Decimal(str(amount))
    if amount <= 0:
        print("Amount 0 se zyada hona chahiye.")
        return
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT balance FROM accounts WHERE account_no = %s FOR UPDATE",
            (account_no,)
        )
        row = cursor.fetchone()
        if row is None:
            print("Ye account exist nahi karta.")
            return
        if row[0] < amount:
            print("Balance kam hai.")
            return
        cursor.execute(
            "UPDATE accounts SET balance = balance - %s WHERE account_no = %s",
            (amount, account_no)
        )
        cursor.execute(
            "INSERT INTO transactions (account_no, type, amount) VALUES (%s, 'withdraw', %s)",
            (account_no, amount)
        )
        conn.commit()
        print("Withdraw ho gaya.")
    except mysql.connector.Error as e:
        conn.rollback()
        print("Error:", e)
    finally:
        conn.close()
deposit(1, 500)
withdraw(1, 200)
withdraw(1, 100000)
deposit(1, -5)

Deposit ho gaya.
Withdraw ho gaya.
Balance kam hai.
Amount 0 se zyada hona chahiye.


In [6]:
def statement(account_no):
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT name FROM accounts WHERE account_no = %s",
            (account_no,)
        )
        if cursor.fetchone() is None:
            print("Ye account exist nahi karta.")
            return
        cursor.execute(
            "SELECT id, type, amount, created_at FROM transactions "
            "WHERE account_no = %s ORDER BY created_at, id",
            (account_no,)
        )
        rows = cursor.fetchall()
        if not rows:
            print("Abhi koi transaction nahi hai.")
            return
        print("ID | Type | Amount | Date")
        for r in rows:
            print(r[0], "|", r[1], "|", r[2], "|", r[3])
    finally:
        conn.close()
statement(1)
statement(999)

ID | Type | Amount | Date
1 | deposit | 500.00 | 2026-10-04 03:46:53
2 | withdraw | 200.00 | 2026-10-04 03:46:53
3 | withdraw | 200.00 | 2026-10-04 03:49:44
4 | deposit | 500.00 | 2026-10-04 03:52:58
5 | withdraw | 200.00 | 2026-10-04 03:52:58
6 | deposit | 500.00 | 2026-10-04 12:27:54
7 | withdraw | 200.00 | 2026-10-04 12:27:54
9 | withdraw | 200.00 | 2026-10-04 12:28:05
10 | deposit | 500.00 | 2026-10-04 12:29:16
11 | withdraw | 200.00 | 2026-10-04 12:29:16
Ye account exist nahi karta.


In [7]:
from decimal import Decimal, InvalidOperation

def read_account_no():
    # user se account number lo, number nahi hai to None return karo
    try:
        return int(input("Account number: "))
    except ValueError:
        print("Account number sirf number hona chahiye.")
        return None

def read_amount(prompt="Amount: "):
    # user se amount lo, number nahi hai to None return karo
    try:
        return Decimal(input(prompt))
    except InvalidOperation:
        print("Amount sirf number hona chahiye.")
        return None

def menu():
    while True:
        print("\n--- Bank Management System ---")
        print("1. Account banao")
        print("2. Account details dekho")
        print("3. Deposit")
        print("4. Withdraw")
        print("5. Statement")
        print("6. Exit")
        choice = input("Choice (1-6): ").strip()

        if choice == "1":
            name = input("Name: ")
            opening = read_amount("Opening balance: ")
            if opening is not None:
                create_account(name, opening)
        elif choice == "2":
            acc = read_account_no()
            if acc is not None:
                view_account(acc)
        elif choice == "3":
            acc = read_account_no()
            amt = read_amount() if acc is not None else None
            if acc is not None and amt is not None:
                deposit(acc, amt)
        elif choice == "4":
            acc = read_account_no()
            amt = read_amount() if acc is not None else None
            if acc is not None and amt is not None:
                withdraw(acc, amt)
        elif choice == "5":
            acc = read_account_no()
            if acc is not None:
                statement(acc)
        elif choice == "6":
            print("Bye!")
            break
        else:
            print("Galat choice. 1 se 6 ke beech number daalo.")

menu()


--- Bank Management System ---
1. Account banao
2. Account details dekho
3. Deposit
4. Withdraw
5. Statement
6. Exit


Choice (1-6):  2
Account number:  1


Account No: 1 | Name: Test User | Balance: 1800.00

--- Bank Management System ---
1. Account banao
2. Account details dekho
3. Deposit
4. Withdraw
5. Statement
6. Exit


Choice (1-6):  2
Account number:  55


Ye account exist nahi karta.

--- Bank Management System ---
1. Account banao
2. Account details dekho
3. Deposit
4. Withdraw
5. Statement
6. Exit


...

Deposit ho gaya.

--- Bank Management System ---
1. Account banao
2. Account details dekho
3. Deposit
4. Withdraw
5. Statement
6. Exit


Choice (1-6):  5
Account number:  1


ID | Type | Amount | Date
1 | deposit | 500.00 | 2026-10-04 03:46:53
2 | withdraw | 200.00 | 2026-10-04 03:46:53
3 | withdraw | 200.00 | 2026-10-04 03:49:44
4 | deposit | 500.00 | 2026-10-04 03:52:58
5 | withdraw | 200.00 | 2026-10-04 03:52:58
6 | deposit | 500.00 | 2026-10-04 12:27:54
7 | withdraw | 200.00 | 2026-10-04 12:27:54
9 | withdraw | 200.00 | 2026-10-04 12:28:05
10 | deposit | 500.00 | 2026-10-04 12:29:16
11 | withdraw | 200.00 | 2026-10-04 12:29:16
12 | deposit | 10.00 | 2026-10-04 12:44:47

--- Bank Management System ---
1. Account banao
2. Account details dekho
3. Deposit
4. Withdraw
5. Statement
6. Exit


Choice (1-6):  6


Bye!
